# Single Agent - Framework




## Install dependencies
Run this once in a fresh environment.


In [1]:
# %pip -q install langchain-openai python-dotenv
# %pip install -U langchain

## 1) Imports

In [2]:
# from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import HumanMessage

c:\Users\chunh\Documents\2606_mobile-class\2-YK_sample_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Load environment variables - please read instructions carefully

In [3]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
# load_dotenv()

In [4]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

## 3) System prompt

In [5]:
SYSTEM = """You are agent with access to weather and currency conversion tools.

Rules:
- Ask clarifying questions only when essential info is missing.
- Do not invent facts. ALWAYS call tools if available.
- If a tool fails, read the tool error carefully, correct the arguments, and retry once. NEVER ask the user — fix it yourself.
- If the retry still fails, explain the issue clearly to the user.

"""


## 4) Tool - estimate trip cost

In [6]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.""" 
    mock_weather = {
        "Tokyo": "Tokyo is 18°C and cloudy.",
        "Singapore": "Singapore is 30°C and partly cloudy.",
        "London": "London is 9°C and rainy."
    }

    return mock_weather.get(city, f"Weather data for {city} is not available.")

In [7]:
@tool
def get_exchange_rate(
    from_currency: str,
    to_currency: str
) -> str:
    """Get the exchange rate between two currencies. eg. 1 USD = 156.5 JPY"""
 
    mock_rates = { 
        ("USD", "JPY"): 156.5,
        ("JPY", "USD"): 0.0064,
        ("USD", "SGD"): 1.34,
        ("SGD", "USD"): 0.75 
    }

    rate = mock_rates.get((from_currency, to_currency))
    
    if rate is None:
        return f"Exchange rate from {from_currency} to {to_currency} is not available."
    return f"1 {from_currency} = {rate} {to_currency}"

In [8]:
class PrintMessagesAfterModel(AgentMiddleware):
    def after_model(self, state, runtime):
        print("\n================= Messages before model call =================")
        for msg in state["messages"]:
            print(f"\n{msg.type}")
            print(msg)
        print("================================================================\n")

In [9]:
class PrintMessagesBeforeModel(AgentMiddleware):
    def before_model(self, state, runtime):
        print("\n================= Messages after model =================")
        for msg in state["messages"]:
            print(f"\n{msg.type}")
            print(msg)
        print("================================================================\n")


## 5) Agent


In [10]:
from langchain_openai import ChatOpenAI

local_llm = ChatOpenAI(
    model="qwen3",                 
    temperature=0,
    openai_api_key="ollama",            
    openai_api_base="http://localhost:11434/v1"
)

agent = create_agent(
    model=local_llm,                    
    tools=[get_weather, get_exchange_rate],
    middleware=[PrintMessagesBeforeModel(), PrintMessagesAfterModel()],
    system_prompt=SYSTEM
)

# agent = create_agent(
#     model="gpt-4.1-mini",
#     tools=[estimate_trip_cost],
#     # middleware=[PrintMessagesBeforeModel(), PrintMessagesAfterModel()],
#     system_prompt=SYSTEM
# )

## 6) Run

In [11]:
msg = "What is the weather in Tokyo and Paris, What's the USD/JPY rate and SGD/Euro rate?"
# agent.invoke({"messages": [{"role": "user", "content": msg}]})
result = agent.invoke({ "messages": [HumanMessage(content=msg)]})


================= Messages after model =================

human
content="What is the weather in Tokyo and Paris, What's the USD/JPY rate and SGD/Euro rate?" additional_kwargs={} response_metadata={} id='3547c3f9-1cbc-4f06-8830-2766c236e627'


================= Messages before model call =================

human
content="What is the weather in Tokyo and Paris, What's the USD/JPY rate and SGD/Euro rate?" additional_kwargs={} response_metadata={} id='3547c3f9-1cbc-4f06-8830-2766c236e627'

ai
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 543, 'prompt_tokens': 303, 'total_tokens': 846, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-469', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f20f6-a984-7553-9eb2-c43c38264c22-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_o97

In [15]:
from pprint import pprint

pprint(result, width=40)

{'messages': [HumanMessage(content="What is the weather in Tokyo and Paris, What's the USD/JPY rate and SGD/Euro rate?", additional_kwargs={}, response_metadata={}, id='3547c3f9-1cbc-4f06-8830-2766c236e627'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 543, 'prompt_tokens': 303, 'total_tokens': 846, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-469', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f20f6-a984-7553-9eb2-c43c38264c22-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_o9773kp3', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'call_kw7fzcuq', 'type': 'tool_call'}, {'name': 'get_exchange_rate', 'args': {'from_currency': 'USD', 'to_currency': 'JPY'}, 'id': 'call_gj2qnvqf', 'type': 'tool_call'}, {